# Movie Recommendation System Demo

This notebook demonstrates the collaborative filtering recommendation system.

In [ ]:
import sys
sys.path.append('../scripts')

import polars as pl
from recommendation.collaborative_filtering import CollaborativeFilteringModel
import json
from pathlib import Path

## 1. Load Your Movie Data

In [ ]:
movies_df = pl.read_parquet('../data/movies_df.parquet')
print(f"Total movies: {len(movies_df)}")
print(f"Liked movies: {movies_df.filter(pl.col('liked') == True).shape[0]}")
print(f"Movies with viewing date: {movies_df.filter(pl.col('viewed').is_not_null()).shape[0]}")

movies_df.head()

## 2. Analyze Recent Viewing History (Last 6 Months)

In [ ]:
from datetime import datetime, timedelta

cutoff_date = datetime.now().date() - timedelta(days=180)

recent_movies = movies_df.filter(
    (pl.col('viewed').is_not_null()) &
    (pl.col('viewed') >= cutoff_date) &
    (pl.col('liked') == True)
)

print(f"Recent liked movies (last 6 months): {len(recent_movies)}")
recent_movies.sort('viewed', descending=True).head(10)

## 3. Extract Viewing Preferences

In [ ]:
from collections import Counter

genres = []
directors = []
for row in recent_movies.iter_rows(named=True):
    if row['genre']:
        genres.extend([g.strip() for g in str(row['genre']).split(',')])
    if row['director']:
        directors.extend([d.strip() for d in str(row['director']).split(',')])

print("Top Genres:")
for genre, count in Counter(genres).most_common(10):
    print(f"  {genre}: {count}")

print("\nTop Directors:")
for director, count in Counter(directors).most_common(5):
    print(f"  {director}: {count}")

print(f"\nAverage Rating: {recent_movies['imdb_rating'].mean():.2f}")

## 4. Train the Recommendation Model

In [ ]:
model = CollaborativeFilteringModel(data_path='../data/movies_df.parquet')
model.train(months_back=6)

## 5. Generate Recommendations

In [ ]:
recommendations = model.predict(top_n=5)

print("\n" + "="*60)
print("TOP 5 MOVIE RECOMMENDATIONS")
print("="*60)

for i, rec in enumerate(recommendations, 1):
    print(f"\n{i}. {rec['title']} ({rec['year']})")
    print(f"   TMDB ID: {rec['tmdb_id']}")
    print(f"   Rating: {rec['rating']:.1f}/10" if rec['rating'] else "   Rating: N/A")
    print(f"   Match Score: {rec['score']}")
    print(f"   Overview: {rec['overview'][:150]}...")
    if rec.get('poster_path'):
        print(f"   Poster: https://image.tmdb.org/t/p/w500{rec['poster_path']}")

## 6. View Saved Recommendations

In [ ]:
latest_rec_path = Path('../data/recommendations/recommendations_latest.json')

if latest_rec_path.exists():
    with open(latest_rec_path, 'r') as f:
        saved_recs = json.load(f)
    
    print(f"Generated at: {saved_recs['generated_at']}")
    print(f"Based on {saved_recs['months_back']} months of viewing history\n")
    
    rec_df = pl.DataFrame(saved_recs['recommendations'])
    display(rec_df)
else:
    print("No saved recommendations found. Run the generate_recommendations.py script first.")

## 7. Visualize Genre Distribution

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

genre_counts = Counter(genres)
top_genres = dict(genre_counts.most_common(10))

plt.figure(figsize=(12, 6))
plt.bar(top_genres.keys(), top_genres.values())
plt.xlabel('Genre')
plt.ylabel('Count')
plt.title('Your Top Genres (Last 6 Months)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Rating Distribution

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(recent_movies['imdb_rating'].to_list(), bins=20, edgecolor='black')
plt.xlabel('IMDB Rating')
plt.ylabel('Frequency')
plt.title('Rating Distribution of Your Recent Liked Movies')
plt.axvline(recent_movies['imdb_rating'].mean(), color='red', linestyle='--', 
            label=f'Mean: {recent_movies["imdb_rating"].mean():.2f}')
plt.legend()
plt.tight_layout()
plt.show()